# 🚀 Distributed Semiconductor Image Restoration on Kaggle (T4 x2 Dual-GPU)

This notebook trains **NAFNet-SR** using **Dual NVIDIA Tesla T4 GPUs (T4 x2)** with **PyTorch DataParallel**, **Automatic Mixed Precision (AMP FP16)**, **Model EMA**, and **Calibrated Composite Metrology Loss**.


## Step 1: Verify Dual GPU (Tesla T4 x2)


In [ ]:
!nvidia-smi
import torch
print('CUDA Available:', torch.cuda.is_available())
print('GPU Count:', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  [+] GPU {i}: {torch.cuda.get_device_name(i)}')


## Step 2: Clone Repository (Branch `Kunal`)


In [ ]:
!git clone -b Kunal https://github.com/kmbeddedd/semicon_2026.git
%cd semicon_2026


## Step 3: Install Dependencies


In [ ]:
!pip install -q -r requirements.txt


## Step 4: Setup Dataset & Validation Split


In [ ]:
import os, glob, shutil, random, zipfile

# Check Kaggle dataset input or local zip
kaggle_inputs = glob.glob('/kaggle/input/**/*.*', recursive=True)
print(f'Found {len(kaggle_inputs)} files in /kaggle/input/')

# If dataset is in kaggle input
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    if 'train' in z.lower():
        print(f'Extracting {z} to data/...')
        os.makedirs('data', exist_ok=True)
        with zipfile.ZipFile(z, 'r') as zip_ref:
            zip_ref.extractall('data')

# Create 10% validation split if not already created
if os.path.exists('data/train/NoisyLR') and not os.path.exists('data/val'):
    random.seed(42)
    os.makedirs('data/val/NoisyLR', exist_ok=True)
    os.makedirs('data/val/GT', exist_ok=True)
    files = sorted(glob.glob('data/train/NoisyLR/*.npy'))
    val_files = random.sample(files, k=int(len(files) * 0.1))
    for f in val_files:
        fname = os.path.basename(f)
        shutil.move(f, os.path.join('data/val/NoisyLR', fname))
        shutil.move(os.path.join('data/train/GT', fname), os.path.join('data/val/GT', fname))
    print(f'Created Val Split: {len(val_files)} samples moved to data/val/')


## Step 5: Train NAFNet-SR on Dual T4 GPUs (Batch Size 64: 32 per GPU)


In [ ]:
!python train.py --epochs 100 --batch_size 64 --lr 8e-4 --warmup_epochs 5 --scale 2 --patch_size 0 --num_workers 4 --save_dir /kaggle/working/weights


## Step 6: Evaluate & Benchmark (Single Pass & 8-Fold TTA)


In [ ]:
# Fast production evaluation (< 14ms)
!python eval.py --input_dir data/val/NoisyLR --target_dir data/val/GT --output_dir /kaggle/working/val_restored --weights /kaggle/working/weights/best_model.pt --scale 2 --batch_size 16 --no_tta --check_clean_damage

# 8-Fold TTA evaluation
!python eval.py --input_dir data/val/NoisyLR --target_dir data/val/GT --output_dir /kaggle/working/val_restored_tta --weights /kaggle/working/weights/best_model.pt --scale 2 --batch_size 16
